In [1]:
import numpy as np
import pandas as pd

In [2]:
l=["st1.txt","st2.txt","st3.txt","st4.txt","st5.txt"]

In [3]:
def data(file_name):
    st="Hi";
    with open(file_name,"r",encoding="utf-8") as f:
        for line in f:
           st+="\n"+line.lower()

    return st

In [4]:
st=""
for file_name in l:
    st+="\n"+data(file_name)

In [5]:
import re
text=""
for sentence in st.split('\n'):
    if "jain:" in sentence:
        start=sentence.find("jain:")
        sentence=sentence[start:]
        sentence=sentence[5:]
        sentence=sentence.strip()
        sentence = re.split(r'\d', sentence, 1)[0].strip()
        text=text+'\n'+sentence

/tmp/ipykernel_1745/3507049385.py:9: DeprecationWarning: 'maxsplit' is passed as positional argument
  sentence = re.split(r'\d', sentence, 1)[0].strip()


In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [7]:
tok=Tokenizer()

In [8]:
tok.fit_on_texts([text])

In [9]:
len(tok.word_index)

4139

In [10]:
input=[]
output=[]
for sentence in text.split("\n"):
    tokenized_sentences=(tok.texts_to_sequences([sentence]))
    if len(tokenized_sentences[0]) !=1 and len(tokenized_sentences[0])!=0 and len(tokenized_sentences[0])<15:
        for i in range(1,len(tokenized_sentences[0])):
            input.append(tokenized_sentences[0][:i])
            output.append(tokenized_sentences[0][i])

In [11]:
len(input)

27547

In [12]:
max_length=0
for i in input:
    max_length=max(max_length,len(i))

max_length

13

In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [14]:
df = pad_sequences(
    input,
    maxlen=max_length,
    padding="pre",
    truncating="pre"
)

In [15]:
df.shape

(27547, 13)

In [16]:
y=np.array(output)

In [17]:
# from sklearn.model_selection import train_test_split

In [18]:
# x_train , x_test , y_train , y_test = train_test_split(df,y,test_size=0.2,random_state=42)

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense

In [20]:
model= Sequential()

In [21]:
model.add(Embedding(input_dim=len(tok.word_index) + 1,output_dim=100))
model.add(LSTM(200))
model.add(Dense(len(tok.word_index) + 1, activation='softmax'))

In [22]:
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [23]:
model.fit(df,y,epochs=100)

Epoch 1/100
861/861 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.1145 - loss: 6.1651
Epoch 2/100
861/861 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.1465 - loss: 5.5710
Epoch 3/100
861/861 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.1824 - loss: 5.1755
Epoch 4/100
861/861 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.2103 - loss: 4.8475
Epoch 5/100
861/861 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.2280 - loss: 4.5651
Epoch 6/100
861/861 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.2450 - loss: 4.3106
Epoch 7/100
861/861 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.2625 - loss: 4.0655
Epoch 8/100
861/861 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.2811 - loss: 3.8254
Epoch 9/100
861/861 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.3063 - loss: 3.5856
Epoch 10/100
861/861 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.3367 - loss: 3.3496
Epoch 11/100
861/861 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.3718 - loss: 3.1219
Epoch 12/100
861/861 ━━━━━━━━━━━━━━━━━━

In [24]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 13, 100)        │       414,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 200)            │       240,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4140)           │       832,140 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,460,822 (17.02 MB)

 Trainable params: 1,486,940 (5.67 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,973,882 (11.34 MB)

In [25]:
def main_func(text):

    for i in range(5):
      token_text=tok.texts_to_sequences([text])[0]
      padded_text=pad_sequences([token_text],maxlen=max_length,padding='pre')
      ans=np.argmax(model.predict(padded_text))

      word = tok.index_word[ans]
      text+=" "+word

    print(text)

In [26]:
def main_func2(text):
    token_text = tok.texts_to_sequences([text])[0]

    if not token_text:
        print("No known words in input")
        return

    padded_text = pad_sequences(
        [token_text],
        maxlen=max_length,
        padding='pre',
        truncating='pre'
    )

    pred = model.predict(padded_text, verbose=0)[0]

    top3_idx = np.argsort(pred)[-3:][::-1]

    for idx in top3_idx:
        if idx in tok.index_word:
            print(tok.index_word[idx], pred[idx])

In [27]:
# y_pred_arr=model.predict(x_test)

In [28]:
# y_pred=[]
# for i in y_pred_arr:
#   ans=np.argmax(i)
#   y_pred.append(ans)

In [29]:
# from sklearn.metrics import accuracy_score

In [30]:
# accuracy_score(y_test,y_pred)

In [31]:
# y_test

In [32]:
# y_pred=np.array(y_pred)

In [33]:
# y_pred

In [34]:
# for ans1,ans2 in zip(y_test,y_pred):
#   for word,index in tok.word_index.items():
#     if index==ans1:
#       print(word);
#   for word,index in tok.word_index.items():
#     if index==ans2:
#       print(word);
#   print("\n")

In [43]:
main_func2("pohoch gayi ho ")

kya 0.66882455
thik 0.1580671
tum 0.036225323


In [36]:
model.save("model.keras")

In [37]:
import pickle

with open("tokenizer2.pkl", "wb") as f:
    pickle.dump(tok, f)

In [1]:
import sys
import tensorflow as tf
import numpy as np
import keras

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("NumPy:", np.__version__)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
TensorFlow: 2.21.0
Keras: 3.13.2
NumPy: 2.4.6
